# Simple Iron Powder Reconstruction Example

This notebook demonstrates a complete workflow for:
1. Loading iron powder data
2. Processing with instrument response
3. Applying Poisson sampling
4. Frame overlap and reconstruction
5. Bragg edge fitting with nbragg

## Study 1: Single Frame vs Pulse Duration
Test reconstruction quality as a function of pulse duration (10-200 µs)

## Study 2: Multiple Frames with 100 µs Pulse
Test reconstruction quality with 1-10 random frames at pulse_duration=100 µs

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from frame_overlap import Data, Reconstruct, Analysis

%matplotlib inline

## Study 1: Single Frame - Pulse Duration Sweep

Test how pulse duration affects reconstruction with a single frame (no overlap)

In [ ]:
# Pulse duration values to test (µs)
pulse_durations = [10, 20, 50, 100, 150, 200]
results_pulse = []

for pulse_dur in pulse_durations:
    print(f"\nProcessing pulse_duration = {pulse_dur} µs...")
    
    try:
        # Create data object
        data = Data(
            signal_file='iron_powder.csv',
            openbeam_file='openbeam.csv',
            flux=5e6,
            duration=0.5,
            freq=20
        )
        
        # Process pipeline
        data.convolute_response(pulse_duration=pulse_dur)
        data.poisson_sample(flux=1e6, freq=20, measurement_time=8.0)  # 8 hours
        data.overlap(kernel=1, total_time=50, mode='equal')  # Single frame
        
        # Reconstruct
        recon = Reconstruct(data, tmin=None, tmax=None)
        recon.filter(kind='wiener', noise_power=1.0)
        
        # Get reconstruction statistics
        stats = recon.get_statistics()
        
        # nbragg fit with iron_with_cellulose model
        analysis = Analysis(
            xs='iron_with_cellulose',
            vary_background=True,
            vary_response=False,
            vary_weights=True
        )
        analysis.set_params(temp={'vary': False})  # Never vary temperature
        
        result = analysis.fit(recon, wlmin=1.0, wlmax=5.0)
        
        # Extract thickness
        thickness = result.params['thickness'].value
        thickness_err = result.params['thickness'].stderr if result.params['thickness'].stderr else 0
        
        # Store results
        results_pulse.append({
            'pulse_duration': pulse_dur,
            'chi2_per_dof': stats['chi2_per_dof'],
            'r_squared': stats['r_squared'],
            'nbragg_redchi': result.redchi,
            'thickness': thickness,
            'thickness_err': thickness_err
        })
        
        print(f"  ✓ χ²/dof: {stats['chi2_per_dof']:.2f}, nbragg χ²: {result.redchi:.4f}, thickness: {thickness:.3f} ± {thickness_err:.3f} cm")
        
    except Exception as e:
        print(f"  ❌ Failed: {e}")
        import traceback
        traceback.print_exc()
        continue

# Convert to DataFrame
results_pulse_df = pd.DataFrame(results_pulse)
print("\nSweep complete!")
print(results_pulse_df)

In [ ]:
# Plot Study 1 results
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Reconstruction chi2/dof
axes[0, 0].plot(results_pulse_df['pulse_duration'], results_pulse_df['chi2_per_dof'], 'o-', linewidth=2, markersize=8)
axes[0, 0].set_xlabel('Pulse Duration (µs)', fontsize=11)
axes[0, 0].set_ylabel('χ²/dof (Reconstruction)', fontsize=11)
axes[0, 0].set_title('Reconstruction Quality vs Pulse Duration (1 frame)', fontsize=12, fontweight='bold')
axes[0, 0].grid(True, alpha=0.3)

# R-squared
axes[0, 1].plot(results_pulse_df['pulse_duration'], results_pulse_df['r_squared'], 's-', color='green', linewidth=2, markersize=8)
axes[0, 1].axhline(y=0, color='gray', linestyle='--', alpha=0.5)
axes[0, 1].set_xlabel('Pulse Duration (µs)', fontsize=11)
axes[0, 1].set_ylabel('R² (Reconstruction)', fontsize=11)
axes[0, 1].set_title('Coefficient of Determination', fontsize=12, fontweight='bold')
axes[0, 1].grid(True, alpha=0.3)

# nbragg reduced chi2
axes[1, 0].plot(results_pulse_df['pulse_duration'], results_pulse_df['nbragg_redchi'], 'd-', color='red', linewidth=2, markersize=8)
axes[1, 0].set_xlabel('Pulse Duration (µs)', fontsize=11)
axes[1, 0].set_ylabel('Reduced χ² (nbragg)', fontsize=11)
axes[1, 0].set_title('nbragg Fit Quality', fontsize=12, fontweight='bold')
axes[1, 0].grid(True, alpha=0.3)

# Thickness
axes[1, 1].errorbar(results_pulse_df['pulse_duration'], results_pulse_df['thickness'], 
                    yerr=results_pulse_df['thickness_err'], fmt='o-', linewidth=2, markersize=8, capsize=5)
axes[1, 1].set_xlabel('Pulse Duration (µs)', fontsize=11)
axes[1, 1].set_ylabel('Thickness (cm)', fontsize=11)
axes[1, 1].set_title('Fitted Sample Thickness', fontsize=12, fontweight='bold')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('pulse_duration_study.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✓ Plot saved as pulse_duration_study.png")

## Study 2: Multiple Frames with 100 µs Pulse

Test reconstruction quality with varying number of frames at fixed pulse_duration=100 µs

In [ ]:
# Number of frames to test
n_frames_values = range(1, 11)  # 1 to 10 frames
pulse_dur = 100  # µs
results_frames = []

for n_frames in n_frames_values:
    print(f"\nProcessing n_frames = {n_frames} (pulse = {pulse_dur} µs)...")
    
    try:
        # Create data object
        data = Data(
            signal_file='iron_powder.csv',
            openbeam_file='openbeam.csv',
            flux=5e6,
            duration=0.5,
            freq=20
        )
        
        # Process pipeline
        data.convolute_response(pulse_duration=pulse_dur)
        data.poisson_sample(flux=1e6, freq=20, measurement_time=8.0)  # 8 hours
        data.overlap(kernel=n_frames, total_time=50, mode='random', kernel_seed=42)
        
        # Reconstruct
        recon = Reconstruct(data, tmin=None, tmax=None)
        recon.filter(kind='wiener', noise_power=1.0)
        
        # Get reconstruction statistics
        stats = recon.get_statistics()
        
        # nbragg fit with iron_with_cellulose model
        analysis = Analysis(
            xs='iron_with_cellulose',
            vary_background=True,
            vary_response=False,
            vary_weights=True
        )
        analysis.set_params(temp={'vary': False})  # Never vary temperature
        
        result = analysis.fit(recon, wlmin=1.0, wlmax=5.0)
        
        # Extract thickness
        thickness = result.params['thickness'].value
        thickness_err = result.params['thickness'].stderr if result.params['thickness'].stderr else 0
        
        # Store results
        results_frames.append({
            'n_frames': n_frames,
            'chi2_per_dof': stats['chi2_per_dof'],
            'r_squared': stats['r_squared'],
            'nbragg_redchi': result.redchi,
            'thickness': thickness,
            'thickness_err': thickness_err
        })
        
        print(f"  ✓ χ²/dof: {stats['chi2_per_dof']:.2f}, nbragg χ²: {result.redchi:.4f}, thickness: {thickness:.3f} ± {thickness_err:.3f} cm")
        
    except Exception as e:
        print(f"  ❌ Failed: {e}")
        import traceback
        traceback.print_exc()
        continue

# Convert to DataFrame
results_frames_df = pd.DataFrame(results_frames)
print("\nSweep complete!")
print(results_frames_df)

In [ ]:
# Plot Study 2 results
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Reconstruction chi2/dof
axes[0, 0].plot(results_frames_df['n_frames'], results_frames_df['chi2_per_dof'], 'o-', linewidth=2, markersize=8)
axes[0, 0].set_xlabel('Number of Frames', fontsize=11)
axes[0, 0].set_ylabel('χ²/dof (Reconstruction)', fontsize=11)
axes[0, 0].set_title('Reconstruction Quality vs Number of Frames (100 µs pulse)', fontsize=12, fontweight='bold')
axes[0, 0].grid(True, alpha=0.3)

# R-squared
axes[0, 1].plot(results_frames_df['n_frames'], results_frames_df['r_squared'], 's-', color='green', linewidth=2, markersize=8)
axes[0, 1].axhline(y=0, color='gray', linestyle='--', alpha=0.5)
axes[0, 1].set_xlabel('Number of Frames', fontsize=11)
axes[0, 1].set_ylabel('R² (Reconstruction)', fontsize=11)
axes[0, 1].set_title('Coefficient of Determination', fontsize=12, fontweight='bold')
axes[0, 1].grid(True, alpha=0.3)

# nbragg reduced chi2
axes[1, 0].plot(results_frames_df['n_frames'], results_frames_df['nbragg_redchi'], 'd-', color='red', linewidth=2, markersize=8)
axes[1, 0].set_xlabel('Number of Frames', fontsize=11)
axes[1, 0].set_ylabel('Reduced χ² (nbragg)', fontsize=11)
axes[1, 0].set_title('nbragg Fit Quality', fontsize=12, fontweight='bold')
axes[1, 0].grid(True, alpha=0.3)

# Thickness
axes[1, 1].errorbar(results_frames_df['n_frames'], results_frames_df['thickness'], 
                    yerr=results_frames_df['thickness_err'], fmt='o-', linewidth=2, markersize=8, capsize=5)
axes[1, 1].set_xlabel('Number of Frames', fontsize=11)
axes[1, 1].set_ylabel('Thickness (cm)', fontsize=11)
axes[1, 1].set_title('Fitted Sample Thickness', fontsize=12, fontweight='bold')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('n_frames_study.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✓ Plot saved as n_frames_study.png")

## Summary

This notebook demonstrated:

### Study 1: Single Frame - Pulse Duration Effects
- Tested pulse durations: 10, 20, 50, 100, 150, 200 µs
- Single frame (no overlap)
- Shows how instrument resolution affects reconstruction and fitting

### Study 2: Multiple Frames at 100 µs Pulse
- Tested 1-10 random frames
- Fixed pulse duration: 100 µs
- Shows trade-off between frame overlap and reconstruction quality

### nbragg Fit Settings (Used in Both Studies)
- **Model**: iron_with_cellulose
- **vary_background**: True
- **vary_response**: False
- **vary_weights**: True
- **vary_temperature**: False (never varied)
- **Wavelength range**: 1-5 Å

### Key Observations
- Longer pulse durations blur the Bragg edges but may improve statistics
- More frames generally improve reconstruction but increase complexity
- The iron_with_cellulose model accounts for cellulose impurities in the sample

### Next Steps
- Try `iron_cellulose_fixed_response` model for faster fitting
- Experiment with different kernel modes: `mode='equal'` or `mode='blue_noise'`
- Test different `noise_power` values for the Wiener filter